# 09 -- Pattern detector validation

Paired script: `analysis/pattern_validation.py` -- Python ports of a first slice of
`CandlestickPatternEngine.mqh`'s pattern predicates (bullish/bearish pin bar including
TASK-017's wick-to-body fix, bullish/bearish engulfing), kept algebraically identical to
the MQL5 source.

**Fixed, 2026-07-21 Codex review finding:** this notebook previously checked only bullish
engulfing in a 3-bar array; with the function's default `trend_lookback=5`, neither pin-bar
predicate could even be evaluated (the fixture was too short), despite a comment claiming an
embedded bullish pin bar. It also never invoked `compare_to_mql5_export`. Both are fixed here:
the pin-bar fixture is now long enough for the default `trend_lookback`, and the MQL5
comparison helper is invoked (against a self-consistent synthetic export, since no real one
exists yet -- see the closing cell).

In [ ]:
import sys
import tempfile
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.pattern_validation import compare_to_mql5_export, detect_all_patterns

In [ ]:
# Bullish engulfing at k=0 (same 2-bar fixture as tests/test_pattern_validation.py).
engulfing_result = detect_all_patterns(
    opens=[99.0, 110.0], highs=[113.0, 111.0], lows=[98.0, 99.0], closes=[112.0, 100.0],
)
print("Engulfing fixture:")
print(engulfing_result)
assert bool(engulfing_result.iloc[0]["bullish_engulfing"]) is True

In [ ]:
# Bullish pin bar at k=0 with the DEFAULT trend_lookback=5 -- needs 6 bars
# (index 0..5), unlike the previous (broken) 3-bar version of this
# notebook. Bar 0 is the same hand-verified pin-bar shape as
# tests/test_pattern_validation.py::test_bullish_pin_bar_detected; bars
# 1-4 are irrelevant filler (only bar 0 and bar 5 affect the result);
# bar 5's close (125) > bar 0's close (118) satisfies the "preceding
# down-move" condition at trend_lookback=5.
pin_bar_result = detect_all_patterns(
    opens=[116.0, 100.0, 100.0, 100.0, 100.0, 124.0],
    highs=[120.0, 101.0, 101.0, 101.0, 101.0, 126.0],
    lows=[100.0, 99.0, 99.0, 99.0, 99.0, 123.0],
    closes=[118.0, 100.0, 100.0, 100.0, 100.0, 125.0],
    trend_lookback=5,
)
print("Pin-bar fixture:")
print(pin_bar_result)
assert bool(pin_bar_result.iloc[0]["bullish_pin_bar"]) is True

## Cross-check against an MQL5 export

`compare_to_mql5_export` is invoked here for real (previously it was never called by this
notebook). Since no MQL5 module in this project exports pattern-detection results to a file
yet, the "export" used below is a SELF-CONSISTENT synthetic one -- the Python results
themselves, re-saved to CSV -- which proves the comparison MECHANISM works (zero
disagreements against itself) without claiming this is a real cross-check against MQL5.

In [ ]:
tmp_dir = Path(tempfile.mkdtemp(prefix="themba_pattern_demo_"))
synthetic_mql5_export = tmp_dir / "mql5_export.csv"
engulfing_result.to_csv(synthetic_mql5_export, index=False)

disagreements = compare_to_mql5_export(engulfing_result, synthetic_mql5_export)
print(f"disagreements vs. self-consistent synthetic export: {len(disagreements)}")
assert len(disagreements) == 0

## Cross-check against a REAL MQL5 detector export: PENDING

No MQL5 module in this project exports pattern-detection results to a file yet -- the
self-comparison above only proves `compare_to_mql5_export`'s join/diff logic is correct, not
that the Python and MQL5 detectors agree on real data. Also note: only 4 of
`CandlestickPatternEngine.mqh`'s 18 pattern functions are ported so far (see
`pattern_validation.py`'s own module docstring for the list of what remains).